# SQL Layer: RavenStack Analytics
**Database:** SQLite (sqlite3 — zero setup, built into Python)
**Goal:** Load raw CSVs into a relational database and run the core analytical queries in SQL before pulling results into pandas.

| Query | SQL features used |
|---|---|
| MRR by month and tier | `SUM`, `CASE WHEN`, `strftime` date functions |
| Churn drivers by tier | `COUNT`, window function (`PARTITION BY`) |
| Unit economics | `AVG`, `JULIANDAY` date arithmetic, filtered aggregation |
| Cohort active rate | CTE (`WITH`), `LEFT JOIN`, conditional aggregation |
| Industry retention | Multi-table `JOIN`, `LEFT JOIN`, `GROUP BY` |

In [1]:
import sqlite3
import pandas as pd

DB_PATH = '../ravenstack.db'
conn = sqlite3.connect(DB_PATH)

# Parse dates then convert to ISO strings — SQLite stores dates as text
def load_csv(path, date_cols):
    df = pd.read_csv(path, parse_dates=date_cols)
    for col in date_cols:
        if col in df.columns:
            df[col] = df[col].dt.strftime('%Y-%m-%d')
    return df

accounts     = load_csv('../data/accounts.csv',      ['signup_date'])
subs         = load_csv('../data/subscriptions.csv', ['start_date', 'end_date'])
churn_events = load_csv('../data/churn_events.csv',  ['churn_date'])

accounts.to_sql('accounts',     conn, if_exists='replace', index=False)
subs.to_sql('subscriptions',    conn, if_exists='replace', index=False)
churn_events.to_sql('churn_events', conn, if_exists='replace', index=False)

tables = conn.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()
print('Database ready.')
for (t,) in tables:
    n = conn.execute(f'SELECT COUNT(*) FROM {t}').fetchone()[0]
    print(f'  {t}: {n:,} rows')


Database ready.
  accounts: 500 rows
  subscriptions: 5,000 rows
  churn_events: 600 rows


## Query 1: Monthly MRR by Plan Tier

Aggregates paid subscription revenue by calendar month and tier. Uses `CASE WHEN` inside `SUM` to split churned and expansion MRR in a single pass — avoids multiple subqueries.

In [2]:
sql_mrr = '''
SELECT
    strftime('%Y-%m', start_date)                                              AS month,
    plan_tier,
    COUNT(DISTINCT account_id)                                                 AS accounts,
    ROUND(SUM(mrr_amount), 0)                                                  AS total_mrr,
    ROUND(SUM(CASE WHEN churn_flag  = 1 THEN mrr_amount ELSE 0 END), 0)       AS churned_mrr,
    ROUND(SUM(CASE WHEN upgrade_flag = 1 THEN mrr_amount ELSE 0 END), 0)      AS expansion_mrr
FROM subscriptions
WHERE is_trial = 0
GROUP BY month, plan_tier
ORDER BY month, plan_tier
'''

mrr_df = pd.read_sql_query(sql_mrr, conn)
print(f'Monthly MRR by tier ({len(mrr_df)} rows). Last 9 rows:')
print(mrr_df.tail(9).to_string(index=False))


Monthly MRR by tier (72 rows). Last 9 rows:
  month  plan_tier  accounts  total_mrr  churned_mrr  expansion_mrr
2024-10      Basic       109    76722.0       6916.0         4864.0
2024-10 Enterprise       128   898485.0      72237.0        97311.0
2024-10        Pro       101   198401.0      20580.0        31164.0
2024-11      Basic       138    97299.0      10070.0        10545.0
2024-11 Enterprise       146  1210318.0     136713.0        75023.0
2024-11        Pro       120   241423.0      23961.0        34055.0
2024-12      Basic       158   144305.0      15352.0         6593.0
2024-12 Enterprise       174  1652695.0     238800.0       118405.0
2024-12        Pro       183   476427.0      39396.0        35182.0


## Query 2: Churn Drivers by Plan Tier

Cross-tabs churn reason codes against plan tier. The window function `SUM(COUNT(*)) OVER (PARTITION BY a.plan_tier)` computes the per-tier total without a subquery, enabling a single-pass percentage calculation.

In [3]:
sql_churn = '''
SELECT
    a.plan_tier,
    c.reason_code,
    COUNT(*)                                                                        AS churn_count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (PARTITION BY a.plan_tier), 1)    AS pct_of_tier
FROM churn_events c
JOIN accounts a ON c.account_id = a.account_id
GROUP BY a.plan_tier, c.reason_code
ORDER BY a.plan_tier, churn_count DESC
'''

churn_df = pd.read_sql_query(sql_churn, conn)
print('Churn reason by plan tier:')
print(churn_df.to_string(index=False))


Churn reason by plan tier:
 plan_tier reason_code  churn_count  pct_of_tier
     Basic     support           37         20.6
     Basic    features           34         18.9
     Basic      budget           30         16.7
     Basic     pricing           29         16.1
     Basic  competitor           25         13.9
     Basic     unknown           25         13.9
Enterprise    features           39         20.0
Enterprise     unknown           35         17.9
Enterprise      budget           34         17.4
Enterprise     pricing           32         16.4
Enterprise  competitor           31         15.9
Enterprise     support           24         12.3
       Pro     support           43         19.1
       Pro    features           41         18.2
       Pro      budget           40         17.8
       Pro  competitor           36         16.0
       Pro     unknown           35         15.6
       Pro     pricing           30         13.3


## Query 3: Unit Economics by Plan Tier

Calculates avg MRR, avg lifespan, and LTV from churned subscriptions only. `JULIANDAY` converts ISO date strings to a floating-point day count; dividing the difference by 30.44 gives calendar months. Active subscriptions are right-censored and excluded.

In [4]:
sql_unit = '''
SELECT
    plan_tier,
    COUNT(*)                                                                                AS churned_subs,
    ROUND(AVG(mrr_amount), 0)                                                              AS avg_mrr,
    ROUND(AVG((JULIANDAY(end_date) - JULIANDAY(start_date)) / 30.44), 1)                 AS avg_lifespan_months,
    ROUND(AVG(mrr_amount) * AVG((JULIANDAY(end_date) - JULIANDAY(start_date)) / 30.44), 0) AS ltv
FROM subscriptions
WHERE churn_flag = 1
  AND is_trial   = 0
  AND end_date IS NOT NULL
GROUP BY plan_tier
ORDER BY ltv DESC
'''

unit_df = pd.read_sql_query(sql_unit, conn)
print('Unit economics (churned subs only — active subs are right-censored):')
print(unit_df.to_string(index=False))


Unit economics (churned subs only — active subs are right-censored):
 plan_tier  churned_subs  avg_mrr  avg_lifespan_months     ltv
Enterprise           146   6345.0                  2.7 17295.0
       Pro           134   1345.0                  2.5  3367.0
     Basic           128    567.0                  3.2  1785.0


## Query 4: Cohort Active Rate by Plan Tier

Uses a CTE (`WITH latest_status`) to collapse each account's subscription history to a single active/inactive flag before joining to the accounts table. This avoids double-counting accounts with multiple subscriptions.

In [5]:
sql_cohort = '''
WITH latest_status AS (
    SELECT
        account_id,
        MAX(CASE WHEN churn_flag = 0 THEN 1 ELSE 0 END) AS is_active
    FROM subscriptions
    WHERE is_trial = 0
    GROUP BY account_id
)
SELECT
    a.plan_tier,
    strftime('%Y-%m', a.signup_date)                                            AS cohort_month,
    COUNT(DISTINCT a.account_id)                                                AS cohort_size,
    SUM(ls.is_active)                                                           AS active_accounts,
    ROUND(SUM(ls.is_active) * 100.0 / COUNT(DISTINCT a.account_id), 1)        AS active_pct
FROM accounts a
LEFT JOIN latest_status ls ON a.account_id = ls.account_id
GROUP BY a.plan_tier, cohort_month
ORDER BY cohort_month, a.plan_tier
'''

cohort_df = pd.read_sql_query(sql_cohort, conn)
print(f'Cohort active rates ({len(cohort_df)} rows). Sample:')
print(cohort_df.head(12).to_string(index=False))


Cohort active rates (72 rows). Sample:
 plan_tier cohort_month  cohort_size  active_accounts  active_pct
     Basic      2023-01            5                5       100.0
Enterprise      2023-01            5                5       100.0
       Pro      2023-01            7                7       100.0
     Basic      2023-02            5                5       100.0
Enterprise      2023-02            3                3       100.0
       Pro      2023-02           10               10       100.0
     Basic      2023-03            7                7       100.0
Enterprise      2023-03            6                6       100.0
       Pro      2023-03            7                7       100.0
     Basic      2023-04            6                6       100.0
Enterprise      2023-04            4                4       100.0
       Pro      2023-04            5                5       100.0


## Query 5: Retention Rate by Industry

Joins accounts to churn_events with a `LEFT JOIN` so accounts with no churn event are retained in the result. `COUNT(DISTINCT c.account_id)` counts only accounts that appear in churn_events; non-churned accounts contribute NULL and are excluded from the count.

In [6]:
sql_industry = '''
SELECT
    a.industry,
    COUNT(DISTINCT a.account_id)                                                            AS total_accounts,
    COUNT(DISTINCT c.account_id)                                                            AS churned_accounts,
    ROUND(COUNT(DISTINCT c.account_id) * 100.0 / COUNT(DISTINCT a.account_id), 1)         AS churn_rate_pct,
    ROUND((1 - COUNT(DISTINCT c.account_id) * 1.0 / COUNT(DISTINCT a.account_id)) * 100, 1) AS retention_pct
FROM accounts a
LEFT JOIN churn_events c ON a.account_id = c.account_id
GROUP BY a.industry
ORDER BY retention_pct DESC
'''

industry_df = pd.read_sql_query(sql_industry, conn)
print('Retention by industry:')
print(industry_df.to_string(index=False))

conn.close()
print('\nConnection closed.')


Retention by industry:
     industry  total_accounts  churned_accounts  churn_rate_pct  retention_pct
   HealthTech              96                64            66.7           33.3
      FinTech             112                76            67.9           32.1
Cybersecurity             100                72            72.0           28.0
       EdTech              79                57            72.2           27.8
     DevTools             113                83            73.5           26.5

Connection closed.
